In [2]:
import pandas as pd
from pathlib import Path

# ============================================================
# 02.15 STEP 1 — CHANNEL & PAYMENT DATA INSPECTION
# ============================================================

BASE_DIR = Path.cwd().parent

SALES_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
)

sales_files = sorted(
    SALES_DIR.glob("sales_*.csv")
)

print("02.15 STEP 1 — CHANNEL & PAYMENT DATA INSPECTION")
print("=" * 70)

print("Sales files found:", len(sales_files))

# ------------------------------------------------------------
# Load all monthly sales files
# ------------------------------------------------------------

sales_list = []

for file in sales_files:
    df = pd.read_csv(file)
    df["source_file"] = file.name
    sales_list.append(df)

sales = pd.concat(
    sales_list,
    ignore_index=True
)

print("Total sales rows:", len(sales))

# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------

required_columns = [
    "sales_line_id",
    "order_id",
    "order_date",
    "customer_id",
    "store_id",
    "sales_channel",
    "payment_method"
]

missing_columns = [
    col
    for col in required_columns
    if col not in sales.columns
]

print("\nRequired Column Check")
print("-" * 70)

print("Missing required columns:", missing_columns)

# ------------------------------------------------------------
# Channel distribution
# ------------------------------------------------------------

print("\nSales Channel Distribution")
print("-" * 70)

print(
    sales["sales_channel"]
    .value_counts(dropna=False)
)

print("\nSales Channel Percentage")
print("-" * 70)

channel_pct = (
    sales["sales_channel"]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .mul(100)
    .round(2)
)

print(channel_pct)

# ------------------------------------------------------------
# Payment method distribution
# ------------------------------------------------------------

print("\nPayment Method Distribution")
print("-" * 70)

print(
    sales["payment_method"]
    .value_counts(dropna=False)
)

print("\nPayment Method Percentage")
print("-" * 70)

payment_pct = (
    sales["payment_method"]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .mul(100)
    .round(2)
)

print(payment_pct)

# ------------------------------------------------------------
# Sales Channel × Payment Method
# ------------------------------------------------------------

print("\nSales Channel × Payment Method")
print("-" * 70)

channel_payment = pd.crosstab(
    sales["sales_channel"],
    sales["payment_method"]
)

print(channel_payment)

# ------------------------------------------------------------
# Channel × Payment percentage
# ------------------------------------------------------------

print("\nSales Channel × Payment Percentage")
print("-" * 70)

channel_payment_pct = (
    pd.crosstab(
        sales["sales_channel"],
        sales["payment_method"],
        normalize="index"
    )
    .mul(100)
    .round(2)
)

print(channel_payment_pct)

# ------------------------------------------------------------
# Store vs Online summary
# ------------------------------------------------------------

print("\nStore vs Online Summary")
print("-" * 70)

channel_summary = (
    sales
    .groupby("sales_channel")
    .agg(
        sales_lines=("sales_line_id", "count"),
        orders=("order_id", "nunique"),
        customers=("customer_id", "nunique"),
        stores=("store_id", "nunique")
    )
    .reset_index()
)

print(
    channel_summary.to_string(index=False)
)

# ------------------------------------------------------------
# Monthly channel behavior
# ------------------------------------------------------------

sales["order_date"] = pd.to_datetime(
    sales["order_date"],
    errors="coerce"
)

sales["month"] = (
    sales["order_date"]
    .dt.to_period("M")
    .astype(str)
)

monthly_channel = (
    sales
    .groupby(
        ["month", "sales_channel"],
        as_index=False
    )
    .agg(
        sales_lines=("sales_line_id", "count"),
        orders=("order_id", "nunique")
    )
)

print("\nMonthly Channel Summary")
print("-" * 70)

print(
    monthly_channel.head(20)
    .to_string(index=False)
)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

invalid_dates = (
    sales["order_date"]
    .isna()
    .sum()
)

missing_channel = (
    sales["sales_channel"]
    .isna()
    .sum()
)

missing_payment = (
    sales["payment_method"]
    .isna()
    .sum()
)

print("\nInspection Validation")
print("-" * 70)

print("Invalid order dates:", invalid_dates)
print("Missing sales channels:", missing_channel)
print("Missing payment methods:", missing_payment)

# ------------------------------------------------------------
# Final Step 1 validation
# ------------------------------------------------------------

if (
    len(sales_files) == 32
    and len(sales) == 2_118_200
    and len(missing_columns) == 0
):
    print("\n02.15 Step 1 inspection: PASSED")
else:
    print("\n02.15 Step 1 inspection: FAILED")

02.15 STEP 1 — CHANNEL & PAYMENT DATA INSPECTION
Sales files found: 32
Total sales rows: 2118200

Required Column Check
----------------------------------------------------------------------
Missing required columns: []

Sales Channel Distribution
----------------------------------------------------------------------
sales_channel
Store     1588625
Online     529575
Name: count, dtype: int64

Sales Channel Percentage
----------------------------------------------------------------------
sales_channel
Store     75.0
Online    25.0
Name: proportion, dtype: float64

Payment Method Distribution
----------------------------------------------------------------------
payment_method
Debit Card     371546
Wallet         370607
UPI            370504
Net Banking    370305
Credit Card    369665
Cash           265573
Name: count, dtype: int64

Payment Method Percentage
----------------------------------------------------------------------
payment_method
Debit Card     17.54
Wallet         17.50
UPI

In [3]:
import pandas as pd
from pathlib import Path

# ============================================================
# 02.15 STEP 2 — CREATE CHANNEL & PAYMENT BEHAVIOR PLAN
# ============================================================

BASE_DIR = Path.cwd().parent

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "channel_payment_adjusted"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# 1. CHANNEL PLAN
# ============================================================

channel_plan = pd.DataFrame({
    "sales_channel": [
        "Store",
        "Online"
    ],
    "target_share_pct": [
        75.0,
        25.0
    ]
})

# ============================================================
# 2. PAYMENT METHOD PLAN
# ============================================================

payment_plan = pd.DataFrame({
    "payment_method": [
        "UPI",
        "Credit Card",
        "Debit Card",
        "Cash",
        "Net Banking"
    ],
    "target_share_pct": [
        35.0,
        25.0,
        20.0,
        10.0,
        10.0
    ]
})

# ============================================================
# 3. CHANNEL × PAYMENT PLAN
# ============================================================

channel_payment_plan = pd.DataFrame({
    "sales_channel": [
        "Store",
        "Store",
        "Store",
        "Store",
        "Store",
        "Online",
        "Online",
        "Online",
        "Online",
        "Online"
    ],
    "payment_method": [
        "UPI",
        "Credit Card",
        "Debit Card",
        "Cash",
        "Net Banking",
        "UPI",
        "Credit Card",
        "Debit Card",
        "Cash",
        "Net Banking"
    ],
    "target_share_within_channel_pct": [
        30.0,
        20.0,
        20.0,
        20.0,
        10.0,
        50.0,
        40.0,
        5.0,
        0.0,
        5.0
    ]
})

# ============================================================
# 4. CUSTOMER BEHAVIOR
# ============================================================

customer_channel_plan = pd.DataFrame({
    "customer_behavior": [
        "Occasional",
        "Regular",
        "Premium",
        "VIP"
    ],
    "online_share_pct": [
        15.0,
        20.0,
        30.0,
        40.0
    ]
})

# ============================================================
# 5. STORE BEHAVIOR
# ============================================================

store_channel_plan = pd.DataFrame({
    "store_behavior": [
        "Neighborhood",
        "High Street",
        "Mall",
        "Outlet"
    ],
    "online_share_pct": [
        15.0,
        20.0,
        30.0,
        25.0
    ]
})

# ============================================================
# 6. PERIOD BEHAVIOR
# ============================================================

period_channel_plan = pd.DataFrame({
    "period_type": [
        "Normal",
        "Festive",
        "Year-End"
    ],
    "online_share_multiplier": [
        1.00,
        1.15,
        1.25
    ]
})

# ============================================================
# VALIDATION
# ============================================================

print("02.15 STEP 2 — CHANNEL & PAYMENT BEHAVIOR PLAN")
print("=" * 70)

# Channel total
channel_total = (
    channel_plan["target_share_pct"]
    .sum()
)

# Payment total
payment_total = (
    payment_plan["target_share_pct"]
    .sum()
)

# Channel × payment totals
channel_payment_totals = (
    channel_payment_plan
    .groupby("sales_channel")[
        "target_share_within_channel_pct"
    ]
    .sum()
)

# Customer validation
customer_values_valid = (
    customer_channel_plan["online_share_pct"]
    .between(0, 100)
    .all()
)

# Store validation
store_values_valid = (
    store_channel_plan["online_share_pct"]
    .between(0, 100)
    .all()
)

# Period validation
period_values_valid = (
    period_channel_plan["online_share_multiplier"]
    > 0
).all()

# ============================================================
# DISPLAY
# ============================================================

print("\nChannel Plan")
print("-" * 70)

print(
    channel_plan.to_string(index=False)
)

print(
    "\nChannel Share Total:",
    channel_total
)

print("\nPayment Method Plan")
print("-" * 70)

print(
    payment_plan.to_string(index=False)
)

print(
    "\nPayment Share Total:",
    payment_total
)

print("\nChannel × Payment Plan")
print("-" * 70)

print(
    channel_payment_plan.to_string(index=False)
)

print("\nChannel × Payment Totals")
print("-" * 70)

print(
    channel_payment_totals
)

print("\nCustomer Channel Behavior")
print("-" * 70)

print(
    customer_channel_plan.to_string(index=False)
)

print("\nStore Channel Behavior")
print("-" * 70)

print(
    store_channel_plan.to_string(index=False)
)

print("\nPeriod Channel Behavior")
print("-" * 70)

print(
    period_channel_plan.to_string(index=False)
)

# ============================================================
# SAVE CONTROL PLANS
# ============================================================

CHANNEL_FILE = (
    OUTPUT_DIR
    / "channel_behavior_plan.csv"
)

PAYMENT_FILE = (
    OUTPUT_DIR
    / "payment_behavior_plan.csv"
)

CHANNEL_PAYMENT_FILE = (
    OUTPUT_DIR
    / "channel_payment_behavior_plan.csv"
)

CUSTOMER_FILE = (
    OUTPUT_DIR
    / "customer_channel_behavior_plan.csv"
)

STORE_FILE = (
    OUTPUT_DIR
    / "store_channel_behavior_plan.csv"
)

PERIOD_FILE = (
    OUTPUT_DIR
    / "period_channel_behavior_plan.csv"
)

channel_plan.to_csv(
    CHANNEL_FILE,
    index=False
)

payment_plan.to_csv(
    PAYMENT_FILE,
    index=False
)

channel_payment_plan.to_csv(
    CHANNEL_PAYMENT_FILE,
    index=False
)

customer_channel_plan.to_csv(
    CUSTOMER_FILE,
    index=False
)

store_channel_plan.to_csv(
    STORE_FILE,
    index=False
)

period_channel_plan.to_csv(
    PERIOD_FILE,
    index=False
)

print("\nFiles saved:")
print(CHANNEL_FILE)
print(PAYMENT_FILE)
print(CHANNEL_PAYMENT_FILE)
print(CUSTOMER_FILE)
print(STORE_FILE)
print(PERIOD_FILE)

# ============================================================
# FINAL VALIDATION
# ============================================================

if (
    channel_total == 100
    and payment_total == 100
    and (
        channel_payment_totals == 100
    ).all()
    and customer_values_valid
    and store_values_valid
    and period_values_valid
):
    print("\n02.15 Step 2 validation: PASSED")
else:
    print("\n02.15 Step 2 validation: FAILED")

02.15 STEP 2 — CHANNEL & PAYMENT BEHAVIOR PLAN

Channel Plan
----------------------------------------------------------------------
sales_channel  target_share_pct
        Store              75.0
       Online              25.0

Channel Share Total: 100.0

Payment Method Plan
----------------------------------------------------------------------
payment_method  target_share_pct
           UPI              35.0
   Credit Card              25.0
    Debit Card              20.0
          Cash              10.0
   Net Banking              10.0

Payment Share Total: 100.0

Channel × Payment Plan
----------------------------------------------------------------------
sales_channel payment_method  target_share_within_channel_pct
        Store            UPI                             30.0
        Store    Credit Card                             20.0
        Store     Debit Card                             20.0
        Store           Cash                             20.0
        Store    Net 

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# 02.15 STEP 3 — APPLY CHANNEL & PAYMENT BEHAVIOR
# ============================================================

BASE_DIR = Path.cwd().parent

SALES_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
)

PLAN_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "channel_payment_adjusted"
)

OUTPUT_DIR = (
    PLAN_DIR
    / "sales_adjusted"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Load behavior plans
# ------------------------------------------------------------

channel_plan = pd.read_csv(
    PLAN_DIR / "channel_behavior_plan.csv"
)

payment_plan = pd.read_csv(
    PLAN_DIR / "payment_behavior_plan.csv"
)

channel_payment_plan = pd.read_csv(
    PLAN_DIR / "channel_payment_behavior_plan.csv"
)

customer_channel_plan = pd.read_csv(
    PLAN_DIR / "customer_channel_behavior_plan.csv"
)

store_channel_plan = pd.read_csv(
    PLAN_DIR / "store_channel_behavior_plan.csv"
)

period_channel_plan = pd.read_csv(
    PLAN_DIR / "period_channel_behavior_plan.csv"
)

# ------------------------------------------------------------
# Load customer and store masters
# ------------------------------------------------------------

customers = pd.read_csv(
    SALES_DIR / "customers_master.csv"
)

stores = pd.read_csv(
    SALES_DIR / "stores_master.csv"
)

# Customer segment lookup
customer_lookup = customers[
    [
        "customer_id",
        "customer_segment"
    ]
].drop_duplicates(
    subset=["customer_id"]
)

# Store type lookup
store_lookup = stores[
    [
        "store_id",
        "store_type"
    ]
].drop_duplicates(
    subset=["store_id"]
)

# ------------------------------------------------------------
# Create lookup dictionaries
# ------------------------------------------------------------

customer_online_share = dict(
    zip(
        customer_channel_plan[
            "customer_behavior"
        ],
        customer_channel_plan[
            "online_share_pct"
        ]
    )
)

store_online_share = dict(
    zip(
        store_channel_plan[
            "store_behavior"
        ],
        store_channel_plan[
            "online_share_pct"
        ]
    )
)

period_multiplier = dict(
    zip(
        period_channel_plan[
            "period_type"
        ],
        period_channel_plan[
            "online_share_multiplier"
        ]
    )
)

# ------------------------------------------------------------
# Payment probabilities
# ------------------------------------------------------------

store_payment_plan = (
    channel_payment_plan[
        channel_payment_plan[
            "sales_channel"
        ] == "Store"
    ]
)

online_payment_plan = (
    channel_payment_plan[
        channel_payment_plan[
            "sales_channel"
        ] == "Online"
    ]
)

store_payment_methods = (
    store_payment_plan[
        "payment_method"
    ].tolist()
)

store_payment_probs = (
    store_payment_plan[
        "target_share_within_channel_pct"
    ].to_numpy(dtype=float)
    / 100
)

online_payment_methods = (
    online_payment_plan[
        "payment_method"
    ].tolist()
)

online_payment_probs = (
    online_payment_plan[
        "target_share_within_channel_pct"
    ].to_numpy(dtype=float)
    / 100
)

# ------------------------------------------------------------
# Monthly sales files
# ------------------------------------------------------------

sales_files = sorted(
    SALES_DIR.glob("sales_*.csv")
)

print("02.15 STEP 3 — APPLY CHANNEL & PAYMENT BEHAVIOR")
print("=" * 70)

print("Sales files:", len(sales_files))

total_rows = 0
total_channel_changes = 0
total_payment_changes = 0

# ------------------------------------------------------------
# Process each monthly file
# ------------------------------------------------------------

for file in sales_files:

    sales = pd.read_csv(file)

    original_channel = sales[
        "sales_channel"
    ].copy()

    original_payment = sales[
        "payment_method"
    ].copy()

    # --------------------------------------------------------
    # Dates
    # --------------------------------------------------------

    sales["order_date"] = pd.to_datetime(
        sales["order_date"],
        errors="coerce"
    )

    sales["month"] = (
        sales["order_date"]
        .dt.month
    )

    # --------------------------------------------------------
    # Customer behavior
    # --------------------------------------------------------

    sales = sales.merge(
        customer_lookup,
        on="customer_id",
        how="left"
    )

    # --------------------------------------------------------
    # Store behavior
    # --------------------------------------------------------

    sales = sales.merge(
        store_lookup,
        on="store_id",
        how="left"
    )

    # --------------------------------------------------------
    # Determine period type
    # --------------------------------------------------------

    sales["period_type"] = "Normal"

    sales.loc[
        sales["month"].isin([9, 10, 11]),
        "period_type"
    ] = "Festive"

    sales.loc[
        sales["month"] == 12,
        "period_type"
    ] = "Year-End"

    # --------------------------------------------------------
    # Customer online share
    # --------------------------------------------------------

    sales["customer_online_share"] = (
        sales["customer_segment"]
        .map(customer_online_share)
        .fillna(20.0)
    )

    # --------------------------------------------------------
    # Store online share
    # --------------------------------------------------------

    sales["store_online_share"] = (
        sales["store_type"]
        .map(store_online_share)
        .fillna(20.0)
    )

    # --------------------------------------------------------
    # Period multiplier
    # --------------------------------------------------------

    sales["period_multiplier"] = (
        sales["period_type"]
        .map(period_multiplier)
        .fillna(1.0)
    )

    # --------------------------------------------------------
    # Combined online probability
    #
    # Customer behavior + store behavior are combined.
    # Period multiplier increases online activity during
    # festive and year-end periods.
    # --------------------------------------------------------

    base_online_share = (
        sales["customer_online_share"]
        + sales["store_online_share"]
    ) / 2

    online_probability = (
        base_online_share
        * sales["period_multiplier"]
    )

    # Keep probability within realistic bounds
    online_probability = (
        online_probability
        .clip(lower=5.0, upper=60.0)
        / 100
    )

    # --------------------------------------------------------
    # Deterministic random generator
    # --------------------------------------------------------

    seed_value = (
        sales["sales_line_id"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .fillna("0")
        .astype(np.int64)
        % 2_147_483_647
    )

    rng = np.random.default_rng(
        int(seed_value.iloc[0])
        if len(seed_value) > 0
        else 42
    )

    random_values = rng.random(
        len(sales)
    )

    # --------------------------------------------------------
    # Apply Store / Online channel
    # --------------------------------------------------------

    sales["sales_channel"] = np.where(
        random_values
        < online_probability,
        "Online",
        "Store"
    )

    # --------------------------------------------------------
    # Generate payment method based on channel
    # --------------------------------------------------------

    store_mask = (
        sales["sales_channel"]
        == "Store"
    )

    online_mask = (
        sales["sales_channel"]
        == "Online"
    )

    store_count = store_mask.sum()
    online_count = online_mask.sum()

    sales.loc[
        store_mask,
        "payment_method"
    ] = rng.choice(
        store_payment_methods,
        size=store_count,
        p=store_payment_probs
    )

    sales.loc[
        online_mask,
        "payment_method"
    ] = rng.choice(
        online_payment_methods,
        size=online_count,
        p=online_payment_probs
    )

    # --------------------------------------------------------
    # Count changes
    # --------------------------------------------------------

    channel_changes = (
        original_channel
        != sales["sales_channel"]
    ).sum()

    payment_changes = (
        original_payment
        != sales["payment_method"]
    ).sum()

    total_channel_changes += channel_changes
    total_payment_changes += payment_changes
    total_rows += len(sales)

    # --------------------------------------------------------
    # Remove helper columns
    # --------------------------------------------------------

    helper_columns = [
        "customer_segment",
        "store_type",
        "month",
        "period_type",
        "customer_online_share",
        "store_online_share",
        "period_multiplier"
    ]

    sales.drop(
        columns=helper_columns,
        inplace=True,
        errors="ignore"
    )

    # --------------------------------------------------------
    # Save adjusted file
    # --------------------------------------------------------

    output_file = (
        OUTPUT_DIR / file.name
    )

    sales.to_csv(
        output_file,
        index=False
    )

    print(
        f"{file.name}: "
        f"rows={len(sales)}, "
        f"channel_changes={channel_changes}, "
        f"payment_changes={payment_changes}"
    )

# ============================================================
# FINAL VALIDATION
# ============================================================

adjusted_files = sorted(
    OUTPUT_DIR.glob("sales_*.csv")
)

print("\n" + "=" * 70)
print("02.15 STEP 3 — FINAL VALIDATION")
print("=" * 70)

print(
    "Adjusted sales files:",
    len(adjusted_files)
)

print(
    "Total rows processed:",
    total_rows
)

print(
    "Total channel changes:",
    total_channel_changes
)

print(
    "Total payment changes:",
    total_payment_changes
)

# ------------------------------------------------------------
# Validate final files
# ------------------------------------------------------------

validation_rows = 0
invalid_channels = 0
invalid_payments = 0
missing_channels = 0
missing_payments = 0

valid_channels = {
    "Store",
    "Online"
}

valid_payments = set(
    payment_plan[
        "payment_method"
    ]
)

for file in adjusted_files:

    df = pd.read_csv(file)

    validation_rows += len(df)

    invalid_channels += (
        ~df["sales_channel"]
        .isin(valid_channels)
    ).sum()

    invalid_payments += (
        ~df["payment_method"]
        .isin(valid_payments)
    ).sum()

    missing_channels += (
        df["sales_channel"]
        .isna()
    ).sum()

    missing_payments += (
        df["payment_method"]
        .isna()
    ).sum()

print("\nValidation Results")
print("-" * 70)

print(
    "Validation rows:",
    validation_rows
)

print(
    "Invalid channels:",
    invalid_channels
)

print(
    "Invalid payment methods:",
    invalid_payments
)

print(
    "Missing channels:",
    missing_channels
)

print(
    "Missing payment methods:",
    missing_payments
)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

if (
    len(adjusted_files) == 32
    and validation_rows == 2_118_200
    and invalid_channels == 0
    and invalid_payments == 0
    and missing_channels == 0
    and missing_payments == 0
):
    print("\n02.15 Step 3 validation: PASSED")
else:
    print("\n02.15 Step 3 validation: FAILED")

02.15 STEP 3 — APPLY CHANNEL & PAYMENT BEHAVIOR
Sales files: 32
sales_2024-01.csv: rows=53200, channel_changes=18957, payment_changes=44316
sales_2024-02.csv: rows=50400, channel_changes=18088, payment_changes=41917
sales_2024-03.csv: rows=56000, channel_changes=19949, payment_changes=46876
sales_2024-04.csv: rows=57100, channel_changes=20304, payment_changes=47695
sales_2024-05.csv: rows=58800, channel_changes=21010, payment_changes=48946
sales_2024-06.csv: rows=60500, channel_changes=21608, payment_changes=50311
sales_2024-07.csv: rows=57700, channel_changes=20555, payment_changes=47964
sales_2024-08.csv: rows=60500, channel_changes=21464, payment_changes=50360
sales_2024-09.csv: rows=64400, channel_changes=23842, payment_changes=53636
sales_2024-10.csv: rows=75600, channel_changes=28049, payment_changes=62948
sales_2024-11.csv: rows=81200, channel_changes=30196, payment_changes=67728
sales_2024-12.csv: rows=89600, channel_changes=34355, payment_changes=74379
sales_2025-01.csv: rows=

In [8]:
# ============================================================
# FINAL VALIDATION
# ============================================================

files_valid = (
    len(sales_files) == 32
)

rows_valid = (
    len(sales) == 2_118_200
)

columns_valid = (
    len(missing_columns) == 0
)

channel_valid = (
    invalid_channels == 0
    and missing_channels == 0
)

payment_valid = (
    invalid_payments == 0
    and missing_payments == 0
)

date_valid = (
    invalid_dates == 0
)

both_channels_present = (
    sales["sales_channel"].nunique() == 2
)

multiple_payment_methods = (
    sales["payment_method"].nunique() >= 2
)

# Raw distributions automatically represent 100%
channel_distribution_total = (
    sales["sales_channel"]
    .value_counts(normalize=True)
    .sum()
    * 100
)

payment_distribution_total = (
    sales["payment_method"]
    .value_counts(normalize=True)
    .sum()
    * 100
)

channel_total_valid = (
    abs(channel_distribution_total - 100) < 0.0001
)

payment_total_valid = (
    abs(payment_distribution_total - 100) < 0.0001
)

print("\nFinal Validation")
print("-" * 70)

print("32 sales files:", files_valid)
print("2,118,200 rows:", rows_valid)
print("Required columns:", columns_valid)
print("Channels valid:", channel_valid)
print("Payment methods valid:", payment_valid)
print("Order dates valid:", date_valid)
print("Both Store and Online present:", both_channels_present)
print(
    "Multiple payment methods present:",
    multiple_payment_methods
)
print(
    "Channel distribution total 100%:",
    channel_total_valid
)
print(
    "Payment distribution total 100%:",
    payment_total_valid
)

# ============================================================
# FINAL RESULT
# ============================================================

if (
    files_valid
    and rows_valid
    and columns_valid
    and channel_valid
    and payment_valid
    and date_valid
    and both_channels_present
    and multiple_payment_methods
    and channel_total_valid
    and payment_total_valid
):
    print("\n02.15 Step 4 validation: PASSED")
else:
    print("\n02.15 Step 4 validation: FAILED")


Final Validation
----------------------------------------------------------------------
32 sales files: True
2,118,200 rows: True
Required columns: True
Channels valid: True
Payment methods valid: True
Order dates valid: True
Both Store and Online present: True
Multiple payment methods present: True
Channel distribution total 100%: True
Payment distribution total 100%: True

02.15 Step 4 validation: PASSED


In [10]:
import pandas as pd
from pathlib import Path

# ============================================================
# 02.15 STEP 5 — FINAL CHANNEL & PAYMENT BEHAVIOR PLAN
# ============================================================

BASE_DIR = Path.cwd().parent

ADJUSTED_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "channel_payment_adjusted"
    / "sales_adjusted"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "channel_payment_adjusted"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Load adjusted sales files
# ------------------------------------------------------------

sales_files = sorted(
    ADJUSTED_DIR.glob("sales_*.csv")
)

print("Adjusted sales files found:", len(sales_files))

if len(sales_files) == 0:
    raise FileNotFoundError(
        f"No sales files found in:\n{ADJUSTED_DIR}"
    )

sales_list = []

for file in sales_files:
    df = pd.read_csv(file)
    sales_list.append(df)

sales = pd.concat(
    sales_list,
    ignore_index=True
)

# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------

required_columns = [
    "order_date",
    "sales_channel",
    "payment_method"
]

missing_columns = [
    col
    for col in required_columns
    if col not in sales.columns
]

# ------------------------------------------------------------
# Channel distribution
# ------------------------------------------------------------

channel_distribution = (
    sales["sales_channel"]
    .value_counts()
    .rename_axis("sales_channel")
    .reset_index(name="sales_lines")
)

channel_distribution["share_pct"] = (
    channel_distribution["sales_lines"]
    / len(sales)
    * 100
)

# ------------------------------------------------------------
# Payment distribution
# ------------------------------------------------------------

payment_distribution = (
    sales["payment_method"]
    .value_counts()
    .rename_axis("payment_method")
    .reset_index(name="sales_lines")
)

payment_distribution["share_pct"] = (
    payment_distribution["sales_lines"]
    / len(sales)
    * 100
)

# ------------------------------------------------------------
# Channel × Payment behavior
# ------------------------------------------------------------

channel_payment_distribution = (
    sales
    .groupby(
        ["sales_channel", "payment_method"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "sales_lines"})
)

channel_payment_distribution["share_within_channel_pct"] = (
    channel_payment_distribution
    .groupby("sales_channel")["sales_lines"]
    .transform(
        lambda x: x / x.sum() * 100
    )
)

# ------------------------------------------------------------
# Save final behavior plan
# ------------------------------------------------------------

FINAL_FILE = (
    OUTPUT_DIR
    / "channel_payment_behavior_plan.csv"
)

channel_payment_distribution.to_csv(
    FINAL_FILE,
    index=False
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

channel_total = (
    channel_distribution["share_pct"].sum()
)

payment_total = (
    payment_distribution["share_pct"].sum()
)

invalid_channel_payment_rows = (
    channel_payment_distribution["sales_lines"] <= 0
).sum()

print("\n02.15 STEP 5 — FINAL CHANNEL & PAYMENT BEHAVIOR PLAN")
print("=" * 70)

print("Sales files:", len(sales_files))
print("Sales rows:", len(sales))
print("Missing required columns:", len(missing_columns))

print("\nChannel Distribution:")
print(
    channel_distribution.to_string(index=False)
)

print("\nPayment Distribution:")
print(
    payment_distribution.to_string(index=False)
)

print("\nChannel × Payment Distribution:")
print(
    channel_payment_distribution.to_string(index=False)
)

print("\nValidation")
print("-" * 70)

print(
    "Channel distribution total:",
    round(channel_total, 2),
    "%"
)

print(
    "Payment distribution total:",
    round(payment_total, 2),
    "%"
)

print(
    "Invalid channel-payment rows:",
    invalid_channel_payment_rows
)

print("\nFinal plan saved to:")
print(FINAL_FILE)

# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

if (
    len(sales_files) == 32
    and len(sales) == 2_118_200
    and len(missing_columns) == 0
    and abs(channel_total - 100) < 0.01
    and abs(payment_total - 100) < 0.01
    and invalid_channel_payment_rows == 0
):
    print("\n02.15 Step 5 validation: PASSED")
else:
    print("\n02.15 Step 5 validation: FAILED")

Adjusted sales files found: 32

02.15 STEP 5 — FINAL CHANNEL & PAYMENT BEHAVIOR PLAN
Sales files: 32
Sales rows: 2118200
Missing required columns: 0

Channel Distribution:
sales_channel  sales_lines  share_pct
        Store      1643085  77.569871
       Online       475115  22.430129

Payment Distribution:
payment_method  sales_lines  share_pct
           UPI       730161  34.470824
   Credit Card       518706  24.488056
    Debit Card       352324  16.633179
          Cash       328627  15.514446
   Net Banking       188382   8.893494

Channel × Payment Distribution:
sales_channel payment_method  sales_lines  share_within_channel_pct
       Online    Credit Card       190575                 40.111341
       Online     Debit Card        23608                  4.968902
       Online    Net Banking        23998                  5.050988
       Online            UPI       236934                 49.868769
        Store           Cash       328627                 20.000609
        Store   

In [11]:
import pandas as pd
from pathlib import Path

# ============================================================
# 02.15 STEP 6 — FINAL CHANNEL & PAYMENT PLAN VALIDATION
# ============================================================

BASE_DIR = Path.cwd().parent

PLAN_FILE = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "channel_payment_adjusted"
    / "channel_payment_behavior_plan.csv"
)

# Load final behavior plan
plan = pd.read_csv(PLAN_FILE)

# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------

required_columns = [
    "sales_channel",
    "payment_method",
    "sales_lines",
    "share_within_channel_pct"
]

missing_columns = [
    col
    for col in required_columns
    if col not in plan.columns
]

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

duplicate_rows = plan.duplicated(
    subset=["sales_channel", "payment_method"]
).sum()

missing_values = (
    plan[required_columns]
    .isna()
    .sum()
    .sum()
)

invalid_sales_lines = (
    plan["sales_lines"] <= 0
).sum()

invalid_shares = (
    plan["share_within_channel_pct"] < 0
).sum()

# ------------------------------------------------------------
# Channel validation
# ------------------------------------------------------------

channels = set(
    plan["sales_channel"].dropna().unique()
)

required_channels = {
    "Store",
    "Online"
}

channel_check = required_channels.issubset(
    channels
)

# ------------------------------------------------------------
# Payment validation
# ------------------------------------------------------------

payment_methods = set(
    plan["payment_method"].dropna().unique()
)

payment_check = (
    len(payment_methods) >= 2
)

# ------------------------------------------------------------
# Payment shares within each channel
# ------------------------------------------------------------

channel_share_totals = (
    plan
    .groupby("sales_channel")[
        "share_within_channel_pct"
    ]
    .sum()
)

share_total_check = (
    (channel_share_totals - 100).abs() < 0.01
).all()

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("02.15 STEP 6 — FINAL CHANNEL & PAYMENT PLAN VALIDATION")
print("=" * 70)

print("Plan rows:", len(plan))
print("Missing required columns:", len(missing_columns))
print("Duplicate channel-payment rows:", duplicate_rows)
print("Missing values:", missing_values)
print("Invalid sales-line values:", invalid_sales_lines)
print("Invalid share values:", invalid_shares)

print("\nChannels:")
print(sorted(channels))

print("\nPayment methods:")
print(sorted(payment_methods))

print("\nPayment share totals by channel:")
print(
    channel_share_totals
)

print("\nValidation")
print("-" * 70)

print(
    "Required channels present:",
    channel_check
)

print(
    "Multiple payment methods present:",
    payment_check
)

print(
    "Payment shares total 100% within each channel:",
    share_total_check
)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

if (
    len(missing_columns) == 0
    and duplicate_rows == 0
    and missing_values == 0
    and invalid_sales_lines == 0
    and invalid_shares == 0
    and channel_check
    and payment_check
    and share_total_check
):
    print("\n02.15 Step 6 validation: PASSED")
else:
    print("\n02.15 Step 6 validation: FAILED")

02.15 STEP 6 — FINAL CHANNEL & PAYMENT PLAN VALIDATION
Plan rows: 9
Missing required columns: 0
Duplicate channel-payment rows: 0
Missing values: 0
Invalid sales-line values: 0
Invalid share values: 0

Channels:
['Online', 'Store']

Payment methods:
['Cash', 'Credit Card', 'Debit Card', 'Net Banking', 'UPI']

Payment share totals by channel:
sales_channel
Online    100.0
Store     100.0
Name: share_within_channel_pct, dtype: float64

Validation
----------------------------------------------------------------------
Required channels present: True
Multiple payment methods present: True
Payment shares total 100% within each channel: True

02.15 Step 6 validation: PASSED
